In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from mord import LogisticAT  # Ordinal logistic regression
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import tensorflow_addons as tfa
from keras import backend as K
from keras import losses
from imblearn.over_sampling import SMOTE, SMOTENC
from tensorflow.keras.layers import Lambda
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
import matplotlib.pyplot as plt
from mord import LogisticAT
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, cohen_kappa_score
import matplotlib.pyplot as plt



# %config Completer.use_jedi = False

In [10]:
r = pd.read_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_nap_1800_100.csv", index_col=0)
print(r.shape)
r.head()

(29708, 282)


,slice_index,filename,start_frame,end_frame,_neutral_mean,browDownLeft_mean,browDownRight_mean,browInnerUp_mean,browOuterUpLeft_mean,browOuterUpRight_mean,...,blinks_median_duration,blinks_std_duration,blinks_min_duration,blinks_max_duration,naps_count,naps_mean_duration,naps_median_duration,naps_std_duration,naps_min_duration,naps_max_duration
0,0,2_21_10_30fps_blendshape.csv,0,1799,0.0,0.020961,0.011672,0.091767,0.175344,0.169944,...,5.0,2.604979,1,14,0,0.0,0.0,0.0,0,0
1,1,2_21_10_30fps_blendshape.csv,100,1899,0.0,0.023722,0.013317,0.084261,0.166261,0.161839,...,5.0,2.259955,1,14,0,0.0,0.0,0.0,0,0
2,2,2_21_10_30fps_blendshape.csv,200,1999,0.0,0.024122,0.013600,0.087417,0.166378,0.161378,...,5.0,2.309985,1,14,0,0.0,0.0,0.0,0,0
3,3,2_21_10_30fps_blendshape.csv,300,2099,0.0,0.024967,0.013989,0.085200,0.162722,0.157400,...,5.0,2.132263,1,14,0,0.0,0.0,0.0,0,0
4,4,2_21_10_30fps_blendshape.csv,400,2199,0.0,0.026233,0.014633,0.082817,0.159100,0.153994,...,5.0,2.070616,1,14,0,0.0,0.0,0.0,0,0


In [11]:
r['target'] = r['filename'].apply(lambda x:int(x.split('_')[2].replace('10','2').replace('0','0').replace('5','1')))
r['fold'] = r['filename'].apply(lambda x: int(x.split('_')[0]))
r['subject_id'] = r['filename'].apply(lambda x: int(x.split('_')[1]))
r['target'].value_counts()

2    10416
1     9782
0     9510
Name: target, dtype: int64

In [64]:

# Define the neural network model for ordinal regression
def build_ordinal_nn(input_shape):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_shape,)))
    
    for u in units:
        model.add(layers.Dense(u, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_ratio))
    
    # Output layer for ordinal regression
    model.add(layers.Dense(1, activation=lambda x: 3 * tf.keras.activations.sigmoid(x) - 1.5))  
    
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=1e-4),
        loss='mse',  # MSE is a common loss for ordinal regression
        metrics=['accuracy']
    )
    return model

# Define XGBoost model
def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test):
    xgb_params = {
        'objective': 'reg:squarederror',  # Using squared error loss for ordinal treatment
        'eval_metric': 'mae',
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 500,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    }
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    return model

# Define LightGBM model
def train_lightgbm(X_train, y_train, X_val, y_val, X_test, y_test):
    lgb_params = {
        'objective': 'regression_l1',  # L1 loss works well for ordinal data
        'metric': 'mae',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'n_estimators': 500
    }
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    return model

In [65]:
# def loss(y_true, y_pred):
#     weights = K.cast(K.abs(K.argmax(y_true, axis=1) - K.argmax(y_pred, axis=1))/(K.int_shape(y_pred)[1] - 1), dtype='float32')
#     return (1.0 + weights) * losses.categorical_crossentropy(y_true, y_pred)

def ordinal_accuracy(y_true, y_pred):
    # Define thresholds for classification
    thresholds = [0.33, 0.66]  # For mapping sigmoid to 0, 1, 2
    y_pred_class = np.digitize(y_pred, thresholds)  # Map sigmoid output to class labels (0, 1, 2)
    return tf.keras.metrics.sparse_categorical_accuracy(y_true, y_pred_class)

def step(val, steps, outputs):
    for i in range(len(steps)-1):
        low = steps[i]
        high = steps[i+1]
        if val >= low and val <= high:
            return outputs[i]
        
def report(model, x, y, output_dict):
    y_pred = model.predict(x)
#     y_pred = [step(v,[0,0.33,0.66,1],[0,1,2]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-1.5,-0.5,0.5,1.5],[-1,0,1]) for v in model.predict(x, verbose=0)]
#     y_pred = [step(v,[-3,-1,1,3],[-2,0,2]) for v in model.predict(x, verbose=0)]
    if output_dict:
        cr = classification_report(list(y), y_pred, output_dict=True)
        cr['confusion_matrix'] = list(confusion_matrix(list(y), y_pred))
        return cr
    
    print(classification_report(list(y), y_pred))
    print(confusion_matrix(list(y), y_pred))

In [66]:
class Model:
    def __init__(self, cv_on='subject_id'):
        
        self.cv_on = cv_on
        self.models = []
        self.histories = []
        self.reports = []
    
    
    def get_reports(self):
        records = []

        for item in self.reports:
            base_info = {'test': item['test']}
            for report_type in ['train_report', 'val_report', 'test_report']:
                report = item.get(report_type, {})
                for category, metrics in report.items():
                    if category == 'confusion_matrix':
                        cm_flat = np.array(metrics).flatten()
                        for i, value in enumerate(cm_flat):
                            records.append({**base_info, 'report_type': report_type, 'category': f'conf_matrix_{i}', 'metric': value})
                    elif isinstance(metrics, dict):
                        for metric, value in metrics.items():
                            records.append({**base_info, 'report_type': report_type, 'category': category, 'metric_name': metric, 'metric_value': value})
                    else:
                        records.append({**base_info, 'report_type': report_type, 'category': 'macro avg', 'metric_name': category, 'metric_value': metrics})

        return pd.DataFrame(records)

        
    
    def test_model(self, data):
        unique = data[self.cv_on].unique()
        for i in unique:
            print("==============================")
            print(f"Test {i}")

            train = data[data[self.cv_on]!=i].reset_index(drop=True)
            test  = data[data[self.cv_on]==i].reset_index(drop=True)

            X_train = train.drop(columns=['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id'])
            X_test  = test.drop(columns=['slice_index', 'filename', 'start_frame', 'end_frame', 'target', 'fold', 'subject_id'])

            y_train = train['target']
            y_test  = test['target']

            scaler  = StandardScaler().fit(X_train)

            X_train = scaler.transform(X_train)
            X_test  = scaler.transform(X_test)

            print(f"train dataset dimension {X_train.shape}")
            print(f"test dataset dimension {X_test.shape}")


            # Initialize and train the ordinal logistic regression model
            model = LogisticAT()
            model.fit(X_train, y_train)
            self.models.append(model)

            train_report = report(model, X_train, y_train, True)
            test_report  = report(model, X_test, y_test, True)

            self.reports.append({'test' : i, 
                                 'train_report' : train_report, 
                                 'test_report' : test_report})

In [53]:
m = Model()
m.test_model(r)

Test 21
train dataset dimension (29218, 278)
test dataset dimension (490, 278)
Test 4
train dataset dimension (29209, 278)
test dataset dimension (499, 278)
Test 7
train dataset dimension (29219, 278)
test dataset dimension (489, 278)
Test 1
train dataset dimension (29220, 278)
test dataset dimension (488, 278)
Test 5
train dataset dimension (29216, 278)
test dataset dimension (492, 278)
Test 20
train dataset dimension (29166, 278)
test dataset dimension (542, 278)
Test 39
train dataset dimension (29380, 278)
test dataset dimension (328, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Test 50
train dataset dimension (29163, 278)
test dataset dimension (545, 278)
Test 59
train dataset dimension (29218, 278)
test dataset dimension (490, 278)
Test 57
train dataset dimension (29205, 278)
test dataset dimension (503, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 6
train dataset dimension (29317, 278)
test dataset dimension (391, 278)
Test 12
train dataset dimension (29254, 278)
test dataset dimension (454, 278)
Test 8
train dataset dimension (29219, 278)
test dataset dimension (489, 278)
Test 3
train dataset dimension (29183, 278)
test dataset dimension (525, 278)
Test 11
train dataset dimension (29189, 278)
test dataset dimension (519, 278)
Test 29
train dataset dimension (29230, 278)
test dataset dimension (478, 278)
Test 25
train dataset dimension (29219, 278)
test dataset dimension (489, 278)
Test 26
train dataset dimension (29101, 278)
test dataset dimension (607, 278)
Test 2
train dataset dimension (29215, 278)
test dataset dimension (493, 278)
Test 9
train dataset dimension (29194, 278)
test dataset dimension (514, 278)
Test 23
train dataset dimension (29229, 278)
test dataset dimension (479, 278)
Test 15
train dataset dimension (29197, 278)
test dataset dimension (511, 278)
Test 32
train dataset dimension (29058, 278)
test dataset

...

Test 49
train dataset dimension (28965, 278)
test dataset dimension (743, 278)
Test 52
train dataset dimension (29219, 278)
test dataset dimension (489, 278)
Test 46
train dataset dimension (29542, 278)
test dataset dimension (166, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Test 47
train dataset dimension (29216, 278)
test dataset dimension (492, 278)
Test 16
train dataset dimension (29256, 278)
test dataset dimension (452, 278)
Test 41
train dataset dimension (29243, 278)
test dataset dimension (465, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Test 54
train dataset dimension (29176, 278)
test dataset dimension (532, 278)
Test 28
train dataset dimension (29212, 278)
test dataset dimension (496, 278)
Test 48
train dataset dimension (29205, 278)
test dataset dimension (503, 278)
Test 10
train dataset dimension (29223, 278)
test dataset dimension (485, 278)
Test 27
train dataset dimension (29189, 278)
test dataset dimension (519, 278)
Test 22
train dataset dimension (29212, 278)
test dataset dimension (496, 278)
Test 19
train dataset dimension (29219, 278)
test dataset dimension (489, 278)
Test 53
train dataset dimension (29205, 278)
test dataset dimension (503, 278)
Test 56
train dataset dimension (29206, 278)
test dataset dimension (502, 278)


In [56]:
df = m.get_reports()
df[(df['metric_name'] == 'accuracy') & (df['report_type'] == 'test_report')]['metric_value'].mean()

0.49633505488240204

In [61]:
df[(df['category'].isin(["conf_matrix_0", "conf_matrix_1", "conf_matrix_2", "conf_matrix_3", "conf_matrix_4", "conf_matrix_5", "conf_matrix_6", "conf_matrix_7", "conf_matrix_8"])) & (df['test']==21)]

,test,report_type,category,metric_name,metric_value,metric
21,21,train_report,conf_matrix_0,NaN,NaN,6304.0
22,21,train_report,conf_matrix_1,NaN,NaN,2859.0
23,21,train_report,conf_matrix_2,NaN,NaN,183.0
24,21,train_report,conf_matrix_3,NaN,NaN,1648.0
25,21,train_report,conf_matrix_4,NaN,NaN,6110.0
26,21,train_report,conf_matrix_5,NaN,NaN,1861.0
27,21,train_report,conf_matrix_6,NaN,NaN,212.0
28,21,train_report,conf_matrix_7,NaN,NaN,3310.0
29,21,train_report,conf_matrix_8,NaN,NaN,6731.0
51,21,test_report,conf_matrix_0,NaN,NaN,0.0


In [67]:
r = pd.read_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_7200_100.csv", index_col=0)
print(r.shape)
r['target'] = r['filename'].apply(lambda x:int(x.split('_')[2].replace('10','2').replace('0','0').replace('5','1')))
r['fold'] = r['filename'].apply(lambda x: int(x.split('_')[0]))
r['subject_id'] = r['filename'].apply(lambda x: int(x.split('_')[1]))
r['target'].value_counts()
m = Model()
m.test_model(r)

(20275, 282)
Test 21
train dataset dimension (19947, 278)
test dataset dimension (328, 278)
Test 4
train dataset dimension (19938, 278)
test dataset dimension (337, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 7
train dataset dimension (19948, 278)
test dataset dimension (327, 278)
Test 1
train dataset dimension (19949, 278)
test dataset dimension (326, 278)
Test 5
train dataset dimension (19945, 278)
test dataset dimension (330, 278)
Test 20
train dataset dimension (19895, 278)
test dataset dimension (380, 278)
Test 39
train dataset dimension (20055, 278)
test dataset dimension (220, 278)
Test 50
train dataset dimension (19892, 278)
test dataset dimension (383, 278)
Test 59
train dataset dimension (19947, 278)
test dataset dimension (328, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 57
train dataset dimension (19934, 278)
test dataset dimension (341, 278)


...

/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 48
train dataset dimension (19934, 278)
test dataset dimension (341, 278)
Test 10
train dataset dimension (19952, 278)
test dataset dimension (323, 278)
Test 27
train dataset dimension (19918, 278)
test dataset dimension (357, 278)
Test 22
train dataset dimension (19941, 278)
test dataset dimension (334, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Test 19
train dataset dimension (19948, 278)
test dataset dimension (327, 278)
Test 53
train dataset dimension (19934, 278)
test dataset dimension (341, 278)
Test 56
train dataset dimension (19935, 278)
test dataset dimension (340, 278)


/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mojtaba.kananisarche/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [68]:
df = m.get_reports()
df[(df['metric_name'] == 'accuracy') & (df['report_type'] == 'test_report')]['metric_value'].mean()

0.4939094450809344